# XLZD MF-GP Workflow

This notebook assumes the CNP stage has already been run and that the aggregated CNP CSVs exist under `data/out/cnp`.

It covers only the MF-GP part of the XLZD RESuM workflow:

1. load the XLZD MF-GP settings
2. fit the MF-GP using the CNP output from training LF + training HF
3. generate grid predictions and validation plots
4. inspect the saved metrics, CSV outputs, and plots inline


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "prepare_resum_data.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not find the XLZD repo root from the current working directory.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT / "src" / "run_mfgp") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src" / "run_mfgp"))

from mfgp_clean_pipeline import load_runtime_config, run_clean_mfgp

CONFIG_PATH = REPO_ROOT / "src" / "xlzd" / "settings.yaml"
CNP_TRAIN_CSV = REPO_ROOT / "data" / "out" / "cnp" / "cnp_xlzd_v1_output_15epochs.csv"
CNP_VALIDATION_CSV = REPO_ROOT / "data" / "out" / "cnp" / "cnp_xlzd_v1_output_validation_15epochs.csv"
ITERATION = 0
GRID_POINTS = 120
PREDICT_CHUNK_SIZE = 20000
RANDOM_STATE = 42

print(f"Repo root: {REPO_ROOT}")
print(f"MF-GP config: {CONFIG_PATH}")
print(f"Training CNP CSV: {CNP_TRAIN_CSV}")
print(f"Validation CNP CSV: {CNP_VALIDATION_CSV}")


## 1. Load And Inspect The Runtime Config

This shows the theta headers and output folders the MF-GP notebook will use.


In [ ]:
runtime = load_runtime_config(CONFIG_PATH)

summary = pd.DataFrame(
    {
        "field": [
            "version",
            "theta_headers",
            "theta_min",
            "theta_max",
            "out_dir_cnp",
            "out_dir_mfgp",
        ],
        "value": [
            runtime.version,
            ", ".join(runtime.theta_headers),
            runtime.theta_min,
            runtime.theta_max,
            str(runtime.out_dir_cnp),
            str(runtime.out_dir_mfgp),
        ],
    }
)
summary


## 2. Fit The MF-GP

This reads the training LF+HF aggregated CNP output CSV from `data/out/cnp`, fits the two-level autoregressive MF-GP, and writes all MF-GP artifacts to `data/out/mfgp`.

The validation-only CNP CSV is passed separately so the notebook can also generate the held-out validation plots.


In [ ]:
mfgp_result = run_clean_mfgp(
    config_path=CONFIG_PATH,
    cnp_csv=CNP_TRAIN_CSV,
    validation_csv=CNP_VALIDATION_CSV,
    iteration=ITERATION,
    grid_points_per_axis=GRID_POINTS,
    random_state=RANDOM_STATE,
    predict_chunk_size=PREDICT_CHUNK_SIZE,
    verbose=True,
)

pd.DataFrame(
    {
        "artifact": [
            "cnp_csv",
            "model_json",
            "metrics_json",
            "prediction_csv",
            "grid_csv",
            "data_plot",
            "mean_std_plot",
            "across_theta_plot",
            "coverage_plot",
            "validation_parity_plot",
        ],
        "path": [
            str(mfgp_result.cnp_csv),
            str(mfgp_result.model_json),
            str(mfgp_result.metrics_json),
            str(mfgp_result.prediction_csv),
            str(mfgp_result.grid_csv),
            str(mfgp_result.data_plot),
            str(mfgp_result.mean_std_plot),
            str(mfgp_result.across_theta_plot) if mfgp_result.across_theta_plot else None,
            str(mfgp_result.coverage_plot) if mfgp_result.coverage_plot else None,
            str(mfgp_result.validation_parity_plot) if mfgp_result.validation_parity_plot else None,
        ],
    }
)


## 3. Inspect Metrics And CSV Outputs


In [ ]:
metrics = json.loads(Path(mfgp_result.metrics_json).read_text())
metrics_df = pd.DataFrame(list(metrics.items()), columns=["metric", "value"])
display(metrics_df)

pred_df = pd.read_csv(mfgp_result.prediction_csv)
grid_df = pd.read_csv(mfgp_result.grid_csv)

print("HF prediction rows:", len(pred_df))
display(pred_df.head())

print("Grid rows:", len(grid_df))
display(grid_df.head())


## 4. Display The Saved Plots


In [ ]:
display(Image(filename=str(mfgp_result.data_plot)))
display(Image(filename=str(mfgp_result.mean_std_plot)))

if mfgp_result.across_theta_plot is not None and Path(mfgp_result.across_theta_plot).exists():
    display(Image(filename=str(mfgp_result.across_theta_plot)))

if mfgp_result.coverage_plot is not None and Path(mfgp_result.coverage_plot).exists():
    display(Image(filename=str(mfgp_result.coverage_plot)))

if mfgp_result.validation_parity_plot is not None and Path(mfgp_result.validation_parity_plot).exists():
    display(Image(filename=str(mfgp_result.validation_parity_plot)))


## 5. Optional: Inspect Per-Theta Validation Plots

If `theta_group_plot_dir` was generated, the next cell lists a few files from that directory.


In [ ]:
if mfgp_result.theta_group_plot_dir is not None and Path(mfgp_result.theta_group_plot_dir).exists():
    theta_plot_dir = Path(mfgp_result.theta_group_plot_dir)
    theta_plots = sorted(theta_plot_dir.glob('*.png'))
    print(f"theta-group plots: {len(theta_plots)}")
    pd.DataFrame({"plot": [p.name for p in theta_plots[:10]]})
else:
    print("No theta-group validation plots were generated.")
